# Taylor Series and π Approximation with SymPy

## Learning objectives

By the end of this notebook you will be able to:

1. Compute symbolic Taylor expansions with `sympy.series`
2. Manipulate series: differentiate, substitute, and remove remainder terms
3. Derive the **Leibniz–Gregory** formula for π from `arctan`
4. Measure convergence speed and understand why the series is slow
5. Apply **Machin's formula** for dramatically faster π approximation
6. Appreciate SymPy's exact arithmetic vs floating-point evaluation

---

### Mathematical background

The Taylor expansion of $\arctan(x)$ around $x = 0$ is:

$$\arctan(x) = x - \frac{x^3}{3} + \frac{x^5}{5} - \frac{x^7}{7} + \cdots = \sum_{k=0}^{\infty} \frac{(-1)^k}{2k+1} x^{2k+1}$$

Setting $x = 1$ gives the **Leibniz–Gregory** series:

$$\frac{\pi}{4} = 1 - \frac{1}{3} + \frac{1}{5} - \frac{1}{7} + \cdots$$

**Machin's formula** (1706) converges much faster:

$$\frac{\pi}{4} = 4\arctan\!\left(\frac{1}{5}\right) - \arctan\!\left(\frac{1}{239}\right)$$

## Setup — install and import SymPy

In [ ]:
# Install SymPy if not already available
import importlib, subprocess, sys
if importlib.util.find_spec('sympy') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'sympy'])

from sympy import (
    symbols, atan, series, diff, simplify, trigsimp,
    Rational, pi, sqrt, Integer,
    pprint, init_printing,
    log as symlog, Abs
)
import sympy

init_printing(use_unicode=True)   # pretty-print in terminal; notebooks use LaTeX
print('SymPy version:', sympy.__version__)

---
## Part 1 — Symbolic Taylor expansion

### Exercise 1.1 — Compute the series

Use `sympy.series(f, x, x0, n)` to expand $\arctan(x)$ around $x_0 = 0$ up to (but not including) order $n = 15$.

> **Expected result:**  
> $x - \dfrac{x^3}{3} + \dfrac{x^5}{5} - \dfrac{x^7}{7} + \dfrac{x^9}{9} - \dfrac{x^{11}}{11} + \dfrac{x^{13}}{13} + O(x^{15})$

In [ ]:
x = symbols('x')

# --- YOUR CODE HERE ---
# s = series(...)
# s

In [ ]:
# Solution 1.1
s = series(atan(x), x, 0, n=15)
s

### Exercise 1.2 — Remove the remainder and verify at 0

Call `.removeO()` on the series to obtain a plain polynomial `p`. Then verify that $p(0) = 0$ by substituting `x = 0`.

In [ ]:
# --- YOUR CODE HERE ---
# p = ...
# print('p(0) =', ...)

In [ ]:
# Solution 1.2
p = s.removeO()
print('p(0) =', p.subs(x, 0))   # Expected: 0
p

### Exercise 1.3 — Verify the derivative

The derivative of $\arctan(x)$ is $\dfrac{1}{1+x^2}$.  
Differentiate `p` with respect to `x` and compare it to the Taylor expansion of $\dfrac{1}{1+x^2}$ at the same order. Their difference should simplify to `0`.

In [ ]:
# --- YOUR CODE HERE ---
# dp = diff(...)
# known = series(...).removeO()
# simplify(dp - known)

In [ ]:
# Solution 1.3
dp    = diff(p, x)
known = series(1 / (1 + x**2), x, 0, n=14).removeO()
result = simplify(dp - known)
print('Difference (should be 0):', result)

---
## Part 2 — Approximating π with the Leibniz–Gregory series

### Exercise 2.1 — First approximation

Substitute $x = 1$ into the polynomial `p` from Exercise 1.2.  
You get a rational approximation of $\pi/4$.  
Multiply by 4 and call `.evalf(50)` to see 50 decimal places.  
Compare to `pi.evalf(50)` — how many digits are correct?

In [ ]:
# --- YOUR CODE HERE ---
# approx = 4 * p.subs(x, 1)
# print(approx.evalf(50))
# print(pi.evalf(50))

In [ ]:
# Solution 2.1
approx_pi = 4 * p.subs(x, 1)
print('Approximation  :', approx_pi.evalf(50))
print('True pi        :', pi.evalf(50))
print()
# The exact rational value
print('Exact rational :', approx_pi)

### Exercise 2.2 — Convergence table

Write a loop over odd orders $n = 1, 3, 5, \ldots, 29$.  
For each order, compute the approximation and count the number of correct decimal digits using:

$$\text{digits} = \lfloor -\log_{10}|\text{approx} - \pi| \rfloor$$

Print a table. Observe how slowly the series converges.

In [ ]:
# --- YOUR CODE HERE ---
# for order in range(1, 31, 2):
#     ...

In [ ]:
# Solution 2.2
def correct_digits(approx):
    error = Abs(approx - pi)
    if error == 0:
        return 50
    return int((-symlog(error, 10)).evalf())

print(f"{'Order':>6}  {'π approximation (20 digits)':>28}  {'Correct digits':>14}")
print('-' * 58)
for order in range(1, 31, 2):
    s_n   = series(atan(x), x, 0, n=order).removeO()
    approx = 4 * s_n.subs(x, 1)
    digits = correct_digits(approx)
    print(f"{order:>6}  {str(approx.evalf(20)):>28}  {digits:>14}")

### Reflection

**Why is the Leibniz series so slow?**  
The general term is $\dfrac{(-1)^k}{2k+1}$. Its absolute value decays like $\dfrac{1}{2k+1}$ — only polynomially fast. After $N$ terms the error is approximately $\dfrac{1}{2N+1}$, so you need about $10^d$ terms to gain $d$ decimal digits. That is impractical for more than a few digits.

---
## Part 3 — Machin's formula (bonus)

$$\frac{\pi}{4} = 4\arctan\!\left(\frac{1}{5}\right) - \arctan\!\left(\frac{1}{239}\right)$$

### Exercise 3.1 — Verify the formula symbolically

Use `simplify` or `trigsimp` to confirm that the expression equals $\pi/4$ exactly in SymPy.  
The result should be `True` when compared to zero.

In [ ]:
# --- YOUR CODE HERE ---
# machin_expr = 4 * atan(Rational(1, 5)) - atan(Rational(1, 239))
# simplify(machin_expr - pi / 4)

In [ ]:
# Solution 3.1
machin_expr = 4 * atan(Rational(1, 5)) - atan(Rational(1, 239))
diff_from_pi4 = simplify(machin_expr - pi / 4)
print('4·arctan(1/5) − arctan(1/239) − π/4 =', diff_from_pi4)  # 0
print('Formula verified:', diff_from_pi4 == 0)

### Exercise 3.2 — Machin convergence table

Repeat the loop from Exercise 2.2, but this time approximate `arctan(1/5)` and `arctan(1/239)` using the truncated Taylor polynomial of order `n`, then apply Machin's formula. Print the same convergence table and compare.

In [ ]:
# --- YOUR CODE HERE ---
# for order in range(1, 31, 2):
#     s_n  = series(atan(x), x, 0, n=order).removeO()
#     a5   = s_n.subs(x, Rational(1, 5))
#     a239 = s_n.subs(x, Rational(1, 239))
#     approx = 4 * (4 * a5 - a239)
#     ...

In [ ]:
# Solution 3.2
print(f"{'Order':>6}  {'π approximation (20 digits)':>28}  {'Correct digits':>14}")
print('-' * 58)
for order in range(1, 31, 2):
    s_n   = series(atan(x), x, 0, n=order).removeO()
    a5    = s_n.subs(x, Rational(1, 5))
    a239  = s_n.subs(x, Rational(1, 239))
    approx = 4 * (4 * a5 - a239)
    digits = correct_digits(approx)
    print(f"{order:>6}  {str(approx.evalf(20)):>28}  {digits:>14}")

### Reflection — why is Machin so much faster?

When you evaluate the Taylor polynomial at $x = \tfrac{1}{5}$, successive terms shrink by a factor of $\left(\tfrac{1}{5}\right)^2 = \tfrac{1}{25}$ per step — **exponential convergence**. After $n$ terms the error is roughly $\dfrac{(1/5)^{2n+1}}{2n+1}$.

For $\arctan(1/239)$ the decay is even faster: $(1/239)^2 \approx 1.75 \times 10^{-5}$ per step.

| Series | Terms for 10 digits | Terms for 20 digits |
|--------|--------------------:|--------------------:|
| Leibniz $x=1$ | ~5 billion | ~50 billion |
| Machin $x=1/5$ | ~7 | ~14 |

---
## Part 4 — Side-by-side comparison (visualisation)

Plot the number of correct digits as a function of order for both methods on the same axes.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

orders = list(range(1, 32, 2))

digits_leibniz = []
digits_machin  = []

for order in orders:
    s_n = series(atan(x), x, 0, n=order).removeO()

    # Leibniz
    approx_l = 4 * s_n.subs(x, 1)
    digits_leibniz.append(correct_digits(approx_l))

    # Machin
    a5   = s_n.subs(x, Rational(1, 5))
    a239 = s_n.subs(x, Rational(1, 239))
    approx_m = 4 * (4 * a5 - a239)
    digits_machin.append(correct_digits(approx_m))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(orders, digits_leibniz, 'o-', label='Leibniz  (x = 1)',       color='steelblue')
ax.plot(orders, digits_machin,  's-', label="Machin   (x = 1/5, 1/239)", color='firebrick')
ax.set_xlabel('Polynomial order n')
ax.set_ylabel('Correct decimal digits of π')
ax.set_title('Convergence of Leibniz vs Machin π approximations')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Summary

| Concept | What you practiced |
|---------|-------------------|
| `sympy.series` | Symbolic Taylor expansion with automatic big-O remainder |
| `.removeO()` | Strip the remainder to get a workable polynomial |
| `diff` + `simplify` | Verify the derivative identity symbolically |
| `.subs` + `.evalf` | Evaluate exactly in rationals, then convert to decimals |
| Leibniz–Gregory | $\pi/4 = \arctan(1)$ — correct but very slowly converging |
| Machin's formula | Decomposes $\arctan(1)$ into two fast-converging series |
| Exact arithmetic | SymPy accumulates no floating-point error until `.evalf()` |

---

### Going further

- **Chudnovsky algorithm** — the series used by modern π-computation records; converges at ~14 digits per term.
- `SymPy.mpmath` — arbitrary-precision numerics built into SymPy, with `mp.nstr(mp.pi, 1000)` for high-precision work.
- **Error bounding** — use SymPy's interval arithmetic or `series` remainder term to prove rigorous error bounds, not just empirical digit counts.